### **1. INSTALL LIBRARIES**

In [1]:
# If you're in Colab, pip is usually fine to run in a cell; in local Jupyter, you may prefer your environment manager.
!pip -q install yfinance requests pandas python-dateutil openai

import os
import re
import json
import textwrap
from dataclasses import dataclass
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests

try:
    import yfinance as yf
except Exception as e:
    yf = None
    print("yfinance import failed:", e)

print("✓ Imports ready")


✓ Imports ready


### **2. Config and Helpers**

In [20]:
# --- Config (strict) + helpers: utcnow(), safe_float() ---

import os
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any

import pandas as pd

# Try to import Colab's secure storage; okay if not running in Colab.
try:
    from google.colab import userdata  # type: ignore
    _HAVE_COLAB_USERDATA = True
except Exception:
    _HAVE_COLAB_USERDATA = False

def _required_secret(colab_key: str, env_key: str) -> str:
    """
    Fetch a secret from Colab userdata (preferred) or environment variable.
    Fail fast (ValueError) if missing or blank.
    """
    val = None
    if _HAVE_COLAB_USERDATA:
        try:
            val = userdata.get(colab_key)  # returns None if not set
        except Exception:
            val = None
    if not val:
        val = os.environ.get(env_key)
    if not val or not str(val).strip():
        raise ValueError(f"Missing required secret: '{colab_key}' (or env '{env_key}')")
    return str(val).strip()

@dataclass
class Config:
    # Required runtime inputs
    symbol: str
    days: int

    # Required secrets (strict: must be present)
    openai_key: str
    newsapi_key: str
    fred_key: str
    alpha_key: str

    # Model preference (can be changed later)
    model: str = "gpt-4o-mini"

    @classmethod
    def from_secrets(cls, symbol: str, days: int, model: str = "gpt-4o-mini") -> "Config":
        """
        Build a Config by pulling keys from Colab userdata or env.
        Example:
            cfg = Config.from_secrets("AAPL", 30)
        """
        return cls(
            symbol=symbol,
            days=days,
            openai_key=_required_secret("OpenAI", "OPENAI_API_KEY"),
            newsapi_key=_required_secret("NewsAPI", "NEWSAPI_KEY"),
            fred_key=_required_secret("FredAPI", "FRED_API_KEY"),
            alpha_key=_required_secret("AlphaVantage", "ALPHAVANTAGE_KEY"),
            model=model,
        )

def utcnow() -> datetime:
    """Timezone-aware current UTC (avoids deprecation warnings)."""
    return datetime.now(timezone.utc)

def safe_float(x: Any, default=None):
    """
    Convert anything number-like (strings, numpy/pandas scalars) to a float.
    Returns `default` if conversion fails or is NaN.
    """
    try:
        val = float(x)
        if pd.isna(val):
            return default
        return val
    except Exception:
        return default

print("✓ Config & helpers ready — use Config.from_secrets('AAPL', 30)")


✓ Config & helpers ready — use Config.from_secrets('AAPL', 30)


In [21]:
cfg = Config.from_secrets("AAPL", 30)
print("Built:", {"symbol": cfg.symbol, "days": cfg.days, "openai": bool(cfg.openai_key)})


Built: {'symbol': 'AAPL', 'days': 30, 'openai': True}


### **3. LLM Wrapper**

In [3]:
# LLM wrapper that uses OpenAI if a key/model is available;
# otherwise falls back to a tiny, deterministic heuristic summarizer.
try:
    from openai import OpenAI
except Exception:
    OpenAI = None  # still fine; we'll degrade gracefully

class LLM:
    def __init__(self, api_key: Optional[str], model: str = "gpt-4o-mini"):
        self.api_key = api_key
        self.model = model

    def chat(self, system: str, user: str) -> str:
        """Send a system+user message to the LLM. If unavailable, return heuristic output."""
        if not self.api_key or OpenAI is None:
            return self._local_heuristic(system, user)

        try:
            client = OpenAI(api_key=self.api_key)
            resp = client.chat.completions.create(
                model=self.model,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
                temperature=0.3,
            )
            return (resp.choices[0].message.content or "").strip()
        except Exception as e:
            # degrade gracefully if the API call fails
            return self._local_heuristic(system, f"{user}\n\n[LLM error: {e}]")

    # --- tiny deterministic fallback so the pipeline still runs offline ---
    def _local_heuristic(self, system: str, user: str) -> str:
        text = f"{system}\n{user}"
        # crude sentence split
        parts = [p.strip() for p in re.split(r"(?<=[.!?])\s+", text) if p.strip()]
        bullets = parts[:5]

        # naive tone guess
        pos = sum(bool(re.search(r"\b(beat|growth|up|strong|record|surge)\b", p, re.I)) for p in parts)
        neg = sum(bool(re.search(r"\b(miss|down|weak|decline|lawsuit|fine)\b", p, re.I)) for p in parts)
        tone = "Neutral"
        if pos > neg: tone = "Slightly Positive"
        elif neg > pos: tone = "Slightly Negative"

        return "Summary:\n- " + "\n- ".join(bullets) + f"\n\nTone: {tone}"

print("✓ LLM wrapper ready")


✓ LLM wrapper ready


### **4. Tools**

In [4]:
# Tools = the agent's "toolbox". Each method is one external data capability.
# All methods are defensive: they catch errors and return empty structures on failure.

class Tools:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    # ---------- Prices & financials (Yahoo Finance) ----------
    def yahoo_prices(self, symbol: str, days: int = 30) -> pd.DataFrame:
        """Download OHLCV price data for the last N days."""
        if yf is None:
            print("yfinance not installed.")
            return pd.DataFrame()

        end = utcnow()
        start = end - timedelta(days=days)
        try:
            df = yf.download(
                symbol,
                start=start.date(),
                end=end.date(),
                progress=False,
                auto_adjust=False  # explicit to avoid FutureWarning & surprises
            )
            if isinstance(df, pd.DataFrame) and not df.empty:
                return df.reset_index()
            return pd.DataFrame()
        except Exception as e:
            print("Yahoo prices error:", e)
            return pd.DataFrame()

    def yahoo_info(self, symbol: str) -> Dict[str, Any]:
        """Grab basic company info (may be sparse for some tickers)."""
        if yf is None:
            return {}
        try:
            t = yf.Ticker(symbol)
            # Prefer modern accessor if available; fallback to legacy
            info = t.get_info() if hasattr(t, "get_info") else getattr(t, "info", {}) or {}
            return info or {}
        except Exception as e:
            print("Yahoo info error:", e)
            return {}

    def yahoo_financials(self, symbol: str) -> Dict[str, pd.DataFrame]:
        """Income statement, balance sheet, cash flow (often sparse)."""
        out: Dict[str, pd.DataFrame] = {}
        if yf is None:
            return out
        try:
            t = yf.Ticker(symbol)
            out["income_stmt"]   = getattr(t, "income_stmt", pd.DataFrame())
            out["balance_sheet"] = getattr(t, "balance_sheet", pd.DataFrame())
            out["cashflow"]      = getattr(t, "cashflow", pd.DataFrame())
        except Exception as e:
            print("Yahoo financials error:", e)
        return out

    # ---------- News (NewsAPI.org) ----------
    def newsapi_search(self, query: str, from_days: int = 14, page_size: int = 20) -> List[Dict[str, Any]]:
        """Fetch recent news articles matching a query."""
        key = self.cfg.newsapi_key
        if not key:
            return []
        url = "https://newsapi.org/v2/everything"
        params = {
            "q": query,
            "from": (utcnow() - timedelta(days=from_days)).date().isoformat(),
            "sortBy": "relevancy",
            "language": "en",
            "pageSize": page_size,
            "apiKey": key,
        }
        try:
            r = requests.get(url, params=params, timeout=20)
            r.raise_for_status()
            data = r.json()
            return data.get("articles", []) or []
        except Exception as e:
            print("NewsAPI error:", e)
            return []

    # ---------- Macro (FRED) ----------
    def fred_series(self, series_id: str) -> pd.DataFrame:
        """Download a macroeconomic time series by ID (e.g., CPIAUCSL, UNRATE)."""
        key = self.cfg.fred_key
        if not key:
            return pd.DataFrame()
        url = "https://api.stlouisfed.org/fred/series/observations"
        params = {"series_id": series_id, "api_key": key, "file_type": "json"}
        try:
            r = requests.get(url, params=params, timeout=20)
            r.raise_for_status()
            obs = r.json().get("observations", [])
            df = pd.DataFrame(obs)
            if not df.empty:
                df["value"] = pd.to_numeric(df["value"], errors="coerce")
                df["date"] = pd.to_datetime(df["date"], errors="coerce")
            return df
        except Exception as e:
            print("FRED error:", e)
            return pd.DataFrame()

    # ---------- Company filings/news (Alpha Vantage) ----------
    def alpha_company_news(self, symbol: str) -> List[Dict[str, Any]]:
        """Alpha Vantage NEWS_SENTIMENT endpoint for a ticker."""
        key = self.cfg.alpha_key
        if not key:
            return []
        url = "https://www.alphavantage.co/query"
        params = {"function": "NEWS_SENTIMENT", "tickers": symbol, "apikey": key}
        try:
            r = requests.get(url, params=params, timeout=20)
            r.raise_for_status()
            return r.json().get("feed", []) or []
        except Exception as e:
            print("Alpha Vantage error:", e)
            return []


### **5. Memory** ###

In [5]:
# --- lightweight persistent memory for each symbol ---
class Memory:
    def __init__(self, symbol: str, root: Path = Path(".mafas_memory")):
        self.root = root
        self.root.mkdir(exist_ok=True)  # create folder if missing
        self.path = self.root / f"{symbol.upper()}.json"
        self.data: Dict[str, Any] = {}

        # load existing memory file if present
        if self.path.exists():
            try:
                self.data = json.loads(self.path.read_text())
            except Exception:
                self.data = {}

    def get(self, key: str, default=None):
        """Retrieve a memory item by key."""
        return self.data.get(key, default)

    def set(self, key: str, value: Any):
        """Save (or update) a memory item and write to disk."""
        self.data[key] = value
        try:
            self.path.write_text(json.dumps(self.data, indent=2))
        except Exception:
            pass  # never crash the agent on disk I/O

print("✓ Memory ready")


✓ Memory ready


### **6. Base Agent and Planner** ###

In [6]:
# --- base agent with shared handles + simple logger ---
class Agent:
    def __init__(self, name: str, llm, tools, memory):
        self.name = name
        self.llm = llm
        self.tools = tools
        self.memory = memory

    def log(self, *a):
        print(f"[{self.name}]", *a)

# --- planner that enumerates the research steps ---
class PlannerAgent(Agent):
    def plan(self, symbol: str, days: int):
        steps = [
            f"Fetch {symbol} prices and compute return/volatility for last {days} days",
            "Ingest recent news; preprocess, classify, extract figures; summarize",
            "Route to earnings / macro / market specialist based on news mix",
            "Draft research note; evaluate; refine using feedback",
            "Save brief takeaways to memory",
        ]
        return steps

print("✓ Base + Planner ready")


✓ Base + Planner ready


### **7. News Pipline** ###

In [23]:
class NewsPipelineAgent(Agent):
    """Ingest → Preprocess → Classify → Extract → Summarize
    Primary: NewsAPI; Fallback: Alpha Vantage NEWS_SENTIMENT.
    """

    def run_pipeline(self, query: str) -> Dict[str, Any]:
        self.log("Fetching news… (NewsAPI → AlphaVantage fallback)")
        articles = self._fetch_news_dual_source(query, from_days=14)

        # Ingest
        texts = [(a.get("title") or "") + ". " + (a.get("description") or "") for a in articles]

        # Preprocess
        self.log("Preprocessing…")
        cleaned = [self._clean(t) for t in texts]

        # Classify
        self.log("Classifying…")
        classes = [self._classify(t) for t in cleaned]

        # Extract
        self.log("Extracting…")
        extracts = [self._extract(t) for t in cleaned]

        # Summarize
        self.log("Summarizing…")
        if cleaned:
            joined = "\n".join([f"[{c}] {e[:200]}" for c, e in zip(classes, cleaned)])
            summary = self.llm.chat(
                "You are a market news summarizer.",
                f"Summarize into 5 concise bullets focused on material, stock-moving items.\n\n{joined}"
            )
        else:
            summary = "(No recent articles available from NewsAPI or Alpha Vantage.)"

        return {
            "articles": articles,
            "cleaned": cleaned,
            "classes": classes,
            "extracts": extracts,
            "summary": summary,
        }

    # ---------- helpers ----------
    def _fetch_news_dual_source(self, query: str, from_days: int = 14) -> List[Dict[str, Any]]:
        # 1) NewsAPI (primary)
        arts = self.tools.newsapi_search(query, from_days=from_days, page_size=20)
        if arts:
            return [
                {
                    "source": (a.get("source") or {}).get("name"),
                    "title": a.get("title"),
                    "description": a.get("description"),
                    "url": a.get("url"),
                    "publishedAt": a.get("publishedAt"),
                    "origin": "newsapi",
                }
                for a in arts if a
            ]

        # 2) Alpha Vantage (fallback)
        av = self.tools.alpha_company_news(query)
        if av:
            norm = []
            for item in av:
                norm.append({
                    "source": item.get("source"),
                    "title": item.get("title"),
                    "description": item.get("summary") or "",
                    "url": item.get("url"),
                    "publishedAt": item.get("time_published"),
                    "origin": "alphavantage",
                })
            return norm

        return []

    def _clean(self, text: str) -> str:
        return re.sub(r"\s+", " ", text).strip()

    def _classify(self, text: str) -> str:
        if re.search(r"\b(earnings|eps|revenue|guidance|quarter)\b", text, re.I):
            return "EARNINGS"
        if re.search(r"\b(lawsuit|regulator|FTC|EC|probe|fine)\b", text, re.I):
            return "REGULATORY"
        if re.search(r"\b(product|launch|partnership|acquisition|merger)\b", text, re.I):
            return "CORPORATE"
        return "GENERAL"

    def _extract(self, text: str) -> Dict[str, Any]:
        nums = re.findall(r"\b\$?\d+(?:\.\d+)?\b", text)
        return {"figures": nums[:5]}

print("✓ News pipeline (with fallback) ready")


✓ News pipeline (with fallback) ready


### **8. Router** ###

In [25]:
class RouterAgent(Agent):
    """
    Simple rule-based router:
      - If any news looks like earnings/corporate → prioritize earnings (+market context)
      - If any news looks regulatory → macro
      - Otherwise → market
    """

    def route(self, classes: List[str]) -> str:
        if not classes:
            return "market"  # default when we have no news
        bag = set(classes)
        if "EARNINGS" in bag or "CORPORATE" in bag:
            return "earnings"
        if "REGULATORY" in bag:
            return "macro"
        return "market"

print("✓ Router ready")


✓ Router ready


### **9. Specialists (Earnings, Macro, Market)** ###

In [29]:
class EarningsAnalyzer(Agent):
    def analyze(self, symbol: str) -> str:
        info = self.tools.yahoo_info(symbol)
        fins = self.tools.yahoo_financials(symbol)

        facts = []
        if info:
            mcap = safe_float(info.get("marketCap"))
            pe = safe_float(info.get("trailingPE")) or safe_float(info.get("forwardPE"))
            facts.append(f"Market Cap: ${int(mcap):,}" if mcap is not None else "Market Cap: N/A")
            facts.append(f"P/E: {pe:.1f}" if pe is not None else "P/E: N/A")
            if "industry" in info: facts.append(f"Industry: {info.get('industry')}")
            if "sector" in info:   facts.append(f"Sector: {info.get('sector')}")

        inc = fins.get("income_stmt", pd.DataFrame())
        if isinstance(inc, pd.DataFrame) and not inc.empty:
            try:
                # Try a couple common column names for revenue; take the latest column
                latest_col = inc.columns[0]
                rev_val = inc.loc["Total Revenue", latest_col] if "Total Revenue" in inc.index else \
                          inc.loc["TotalRevenue", latest_col]     if "TotalRevenue" in inc.index else None
                rev_val = safe_float(rev_val)
                if rev_val is not None:
                    facts.append(f"Latest reported revenue: ${int(rev_val):,}")
            except Exception:
                pass

        prompt = (
            "Earnings & Fundamentals:\n- " + "\n- ".join(facts) +
            "\nProvide 4–6 bullets on quality of earnings, margins, and valuation. Include caveats."
        )
        return self.llm.chat("You are an equity analyst.", prompt)


class MacroAnalyzer(Agent):
    def analyze(self, symbol: str) -> str:
        cpi = self.tools.fred_series("CPIAUCSL")
        unemp = self.tools.fred_series("UNRATE")
        macro_note = ""

        if isinstance(cpi, pd.DataFrame) and not cpi.empty:
            cpi_recent = cpi.tail(12)["value"].pct_change().mean() * 100
            cpi_recent = safe_float(cpi_recent)
            if cpi_recent is not None:
                macro_note += f"Avg monthly CPI change (last 12): {cpi_recent:.2f}%\n"

        if isinstance(unemp, pd.DataFrame) and not unemp.empty:
            un = unemp.tail(1)["value"].iloc[0]
            un = safe_float(un)
            if un is not None:
                macro_note += f"Latest unemployment rate: {un:.2f}%\n"

        prompt = f"Relate macro trends to {symbol}. Be specific but concise.\n{macro_note}"
        return self.llm.chat("You are a macro-to-equity strategist.", prompt)


class MarketAnalyzer(Agent):
    def analyze(self, symbol: str, days: int) -> str:
        df = self.tools.yahoo_prices(symbol, days)
        if not isinstance(df, pd.DataFrame) or df.empty:
            return "Price data unavailable."

        # robustly pick a close series
        if "Close" in df.columns:
            close = df["Close"]
        else:
            cand = df.filter(regex=r"(?i)^close$")
            if cand.shape[1] == 0:
                cand = df.filter(regex=r"(?i)close")
            if cand.shape[1] == 0:
                return "Close price not found."
            close = cand.iloc[:, 0]

        close = close.dropna()
        if close.size < 2:
            return "Insufficient price history."

        ret = (close.iloc[-1] / close.iloc[0]) - 1
        vol = close.pct_change().std() * (252 ** 0.5)

        ret_f = safe_float(ret, 0.0)
        vol_f = safe_float(vol, 0.0)

        prompt = (
            f"Analyze {symbol} price action over {days} days.\n"
            f"Return: {ret_f:.2%}, Annualized vol (approx): {vol_f:.2%}.\n"
            "Characterize trend, momentum, and risk in 3–5 bullets."
        )
        return self.llm.chat("You are a technical/market analyst.", prompt)

print("✓ Specialists ready")


✓ Specialists ready


### **10. Evaluator and Optimizer** ###

In [31]:
class EvaluatorOptimizer(Agent):
    def evaluate(self, draft: str) -> Tuple[int, str]:
        """Ask the LLM to score and critique the draft with a strict return format."""
        rubric = textwrap.dedent("""
            Score 1-10 on: accuracy, coherence, use of data, actionability, risk disclosure.
            Provide a one-paragraph critique with concrete suggestions.
            Return format exactly:
            SCORE: <int>
            FEEDBACK: <text>
        """).strip()

        out = self.llm.chat(
            "You are a tough research QA reviewer.",
            f"{rubric}\n\nDRAFT:\n{draft}"
        )

        # robust parse of the required fields
        m = re.search(r"SCORE:\s*(\d+)\s*FEEDBACK:\s*(.*)$", out, re.S | re.I)
        if m:
            try:
                score = int(m.group(1))
                feedback = m.group(2).strip()
                return score, feedback
            except Exception:
                pass
        # fallback if the model didn't follow format perfectly
        return 6, out.strip()

    def optimize(self, draft: str, feedback: str) -> str:
        """Revise the draft guided by feedback; preserve factual content."""
        prompt = textwrap.dedent(f"""
            Improve the research note using this feedback. Preserve factual content,
            add missing caveats, and strengthen conclusions. Keep it concise.

            FEEDBACK:
            {feedback}

            DRAFT:
            {draft}
        """).strip()

        return self.llm.chat("You revise research based on critique.", prompt)

print("✓ Evaluator/Optimizer ready")


✓ Evaluator/Optimizer ready


### **11. Orchestrator** ###

In [33]:
class Orchestrator:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.tools = Tools(cfg)
        self.llm = LLM(cfg.openai_key, cfg.model)
        self.memory = Memory(cfg.symbol)

        self.planner = PlannerAgent("Planner", self.llm, self.tools, self.memory)
        self.news = NewsPipelineAgent("NewsPipeline", self.llm, self.tools, self.memory)
        self.router = RouterAgent("Router", self.llm, self.tools, self.memory)
        self.earn = EarningsAnalyzer("Earnings", self.llm, self.tools, self.memory)
        self.macro = MacroAnalyzer("Macro", self.llm, self.tools, self.memory)
        self.market = MarketAnalyzer("Market", self.llm, self.tools, self.memory)
        self.evalopt = EvaluatorOptimizer("EvalOpt", self.llm, self.tools, self.memory)

    def run(self) -> Dict[str, Any]:
        # 1) Plan
        plan = self.planner.plan(self.cfg.symbol, self.cfg.days)

        # 2) News pipeline
        news_out = self.news.run_pipeline(self.cfg.symbol)

        # 3) Route
        route = self.router.route(news_out.get("classes", []))

        # 4) Specialist analysis
        earn_note = self.earn.analyze(self.cfg.symbol) if route == "earnings" else ""
        macro_note = self.macro.analyze(self.cfg.symbol) if route == "macro" else ""
        # we still want some market context for earnings-heavy days
        market_needed = (route in ("market", "earnings"))
        market_note = self.market.analyze(self.cfg.symbol, self.cfg.days) if market_needed else ""

        # 5) Synthesize draft research note
        draft = self._synthesize(
            self.cfg.symbol,
            plan,
            news_out.get("summary", ""),
            earn_note,
            macro_note,
            market_note
        )

        # 6) Evaluate & refine
        score, feedback = self.evalopt.evaluate(draft)
        refined = self.evalopt.optimize(draft, feedback)

        # 7) Memory update
        self.memory.set("last_route", route)
        self.memory.set("last_score", score)

        return {
            "plan": plan,
            "news": news_out,
            "route": route,
            "earn": earn_note,
            "macro": macro_note,
            "market": market_note,
            "draft": draft,
            "score": score,
            "feedback": feedback,
            "refined": refined,
        }

    def _synthesize(
        self,
        symbol: str,
        plan: List[str],
        news_summary: str,
        earn: str,
        macro: str,
        market: str
    ) -> str:
        plan_text = "- " + "\n- ".join(plan)
        prompt = f"""
Create a concise but professional Research Note for {symbol} with sections:
1) Setup – list the plan bullets
2) News Takeaways – condensed points
3) Earnings & Fundamentals – key points
4) Macro Context – only if relevant
5) Price Action – salient stats
6) Risks & Watchlist – bullet risks + next steps
Aim for ~350–500 words.

PLAN:
{plan_text}

NEWS SUMMARY:
{news_summary}

EARNINGS:
{earn}

MACRO:
{macro}

MARKET:
{market}
""".strip()
        return self.llm.chat("You are a buy-side research writer.", prompt)

print("✓ Orchestrator ready")


✓ Orchestrator ready


### **12. Runner (one-shot execution + pretty print + artifacts)** ###

In [35]:
from pathlib import Path
import json

def run_once(cfg: Config, save_artifacts: bool = True) -> Dict[str, Any]:
    orch = Orchestrator(cfg)
    try:
        result = orch.run()
    except Exception as e:
        print("Run failed:", e)
        return {"error": str(e)}

    # ---- Pretty print core sections ----
    print("\n===== PLAN =====")
    for step in result["plan"]:
        print("-", step)

    print("\n===== ROUTE =====")
    print(result["route"])

    print("\n===== NEWS (summary) =====")
    print(result["news"].get("summary", ""))

    if result.get("earn"):
        print("\n===== EARNINGS =====")
        print(result["earn"])
    if result.get("macro"):
        print("\n===== MACRO =====")
        print(result["macro"])
    if result.get("market"):
        print("\n===== MARKET =====")
        print(result["market"])

    print("\n===== DRAFT =====\n")
    print(result["draft"])

    print("\nScore:", result["score"])
    print("\nFeedback:\n", result["feedback"])

    print("\n===== FINAL (refined) =====\n")
    print(result["refined"])

    # ---- Save compact artifacts (optional) ----
    if save_artifacts:
        out = {
            k: (v if k != "news" else {
                "summary": v.get("summary"),
                "n_articles": len(v.get("articles", []))
            })
            for k, v in result.items()
        }
        out_path = Path(f"mafas_{cfg.symbol}_{utcnow().strftime('%Y%m%d_%H%M%S')}.json")
        out_path.write_text(json.dumps(out, indent=2))
        print(f"\nArtifacts saved to {out_path}")

    return result

print("✓ Runner ready — call run_once(cfg)")


✓ Runner ready — call run_once(cfg)


### **TEST RUN** ###

In [39]:
# Change these two and re-run the cell to analyze a different stock / window.
SYMBOL = "NVDA"   # e.g., "NVDA", "MSFT", "TSLA"
DAYS   = 30       # lookback window for price/news context

# Build a strict Config from your stored secrets (Colab userdata/env)
cfg = Config.from_secrets(SYMBOL, DAYS)

# Fire the pipeline
res = run_once(cfg, save_artifacts=True)

# (Optional) peek at a few convenience fields
print("\n— Convenience —")
print("Plan steps:", len(res.get("plan", [])))
print("Articles:", len(res.get("news", {}).get("articles", [])))
print("Route:", res.get("route"))
print("Score:", res.get("score"))


[NewsPipeline] Fetching news… (NewsAPI → AlphaVantage fallback)
[NewsPipeline] Preprocessing…
[NewsPipeline] Classifying…
[NewsPipeline] Extracting…
[NewsPipeline] Summarizing…


/tmp/ipython-input-490756523.py:76: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  val = float(x)



===== PLAN =====
- Fetch NVDA prices and compute return/volatility for last 30 days
- Ingest recent news; preprocess, classify, extract figures; summarize
- Route to earnings / macro / market specialist based on news mix
- Draft research note; evaluate; refine using feedback
- Save brief takeaways to memory

===== ROUTE =====
earnings

===== NEWS (summary) =====
- **NVIDIA's Market Dominance**: NVIDIA (NVDA) is leading the AI hardware market with strong data center sales and strategic partnerships, positioning it favorably against AMD.
  
- **Analyst Optimism**: Analysts are bullish on NVIDIA, with HSBC projecting an 80% upside and Citi raising its price target, citing the company's pivotal role in the AI semiconductor space.

- **AI Innovations**: Planet Labs is leveraging NVIDIA chips for its new satellite technology, showcasing the practical applications of NVIDIA's AI advancements.

- **Investment Activity**: NVIDIA-backed AI startup n8n has raised $180 million, indicating robust 